# Brand Atlas — 知识图谱阶段性结果查看

本 Notebook 从 PostgreSQL 导出的 JSON 渲染**交互式知识图**（可缩放、拖拽、按类型筛选）。

## 使用前提
1. 已启动 PostgreSQL（`docker compose up -d`）
2. 已跑过 L2/L3 执行器，库里有实体/关系
3. 已导出 JSON：`python -m runtime.visualize.export`

如果还没有导出，先运行下面第 1 个代码块，或到终端执行导出命令。

In [ ]:
# 可选：如果还没导出，直接在这里导出（需要 PostgreSQL 在跑）
import sys, os
sys.path.insert(0, os.path.abspath('../..'))  # 使 runtime 可导入

# 取消注释以下行来导出全部
# from runtime.visualize import export
# with export.DB() as db:
#     graph = export.export_all(db, out_path='output/knowledge_graph.json')
# print('已导出', graph['stats'])

In [ ]:
# 1. 加载上一步导出的 JSON
import json
from pathlib import Path

GRAPH_PATH = Path('output/knowledge_graph.json')
if not GRAPH_PATH.exists():
    raise FileNotFoundError(f'未找到 {GRAPH_PATH}。请先运行 python -m runtime.visualize.export')

graph = json.loads(GRAPH_PATH.read_text(encoding='utf-8'))
print('图谱统计:', graph['stats'])

# 按类型统计节点
from collections import Counter
type_counts = Counter(n['type'] for n in graph['nodes'])
print('\n实体类型分布:', dict(type_counts))
rel_counts = Counter(e['type'] for e in graph['edges'])
print('关系类型分布:', dict(rel_counts))

In [ ]:
# 2. 用 networkx + pyvis 渲染交互式知识图
import networkx as nx
from pyvis.network import Network

G = nx.DiGraph()

# 节点
for n in graph['nodes']:
    G.add_node(n['id'], label=n['label'], ntype=n['type'], status=n.get('status'))

# 关系
for e in graph['edges']:
    if e['subject_id'] in G and e['object_id'] in G:
        G.add_edge(e['subject_id'], e['object_id'], rtype=e['type'],
                   confidence=e.get('confidence'))

# 实体类型的颜色映射
TYPE_COLORS = {
    'brand': '#4e79a7', 'product': '#59a14f', 'organization': '#f28e2b',
    'category': '#e15759', 'industry': '#b07aa1', 'audience': '#76b7b2',
    'problem': '#edc948', 'use_case': '#ff9da7', 'capability': '#9c755f',
    'decision_factor': '#a0a0a0', 'topic': '#86bcb6', 'competitor': '#d4a6c8',
    'content': '#cab2d6', 'assertion': '#8c6d31', 'source': '#6b6b6b',
}

net = Network(height='800px', width='100%', directed=True, notebook=True)
net.barnes_hut()
for nid, data in G.nodes(data=True):
    color = TYPE_COLORS.get(data['ntype'], '#999999')
    net.add_node(nid, label=data['label'], title=f"{data['label']}\n({data['ntype']})",
                 color=color, group=data['ntype'])
for sid, oid, data in G.edges(data=True):
    net.add_edge(sid, oid, title=f"{data['rtype']}", label=data['rtype'])

net.show('output/knowledge_graph.html', notebook=False)
print('已生成交互式图: output/knowledge_graph.html')
print('提示: 在下方使用 IFrame 内嵌显示，或用浏览器打开 HTML')

In [ ]:
# 3. 在 Notebook 内嵌入交互式图（可缩放/拖拽）
from IPython.display import IFrame, display

display(IFrame(src='output/knowledge_graph.html', width='100%', height='800px'))

In [ ]:
# 4. 查看 Assertion / Statement 列表（有证据的陈述）
import pandas as pd

if graph.get('statements'):
    df = pd.DataFrame(graph['statements'])
    print(df[['text', 'class', 'status']].to_string(index=False))
else:
    print('当前导出的数据中没有 statement/assertion。')
    print('先跑 L2 extraction 或 L3 candidate_extraction 后再导出。')

## 说明
**颜色含义**：brand=蓝、product=绿、organization=橙、category=红、industry=紫、audience=青、problem=黄、capability=棕、其余=灰。

**重新导出**：每次跑完执行器后，回到终端运行 `python -m runtime.visualize.export`（或 `--brand <id>`/`--industry <id>` 只看局部），再重新运行本 Notebook 的代码块 2-4 即可刷新图谱。